# Figure 2C — allelic series at repeat loci (rarefaction + PLVI-ranked examples)

Per-locus repeat dosage from the SNV/indel and SV callsets (no TRGT calls): every
length-changing record inside a TRExplorer v1.0.1 locus adds its signed length change to
that haplotype's locus dosage. Both phases use the same inputs and rules: unphased
DeepVariant joint calls for changes < 50 bp and the SV callset for ≥ 50 bp (Phase 1 SVs
start at 50 bp), so an event called by both counts once. Panel C compares how many distinct alleles per
locus Phase 1 and Phase 2 detect, and shows three example loci ranked by PLVI (Danzi et al. 2025).

Three workflows (see [`repeat_loci/README.md`](../../repeat_loci/README.md)), launched from the
Terra Workflows page with the input JSONs this notebook writes:

1. `TrgtLocusConcordance` — integrated locus dosage vs TRGT on chr20 for a participant subset
   (split / double-counted records).
2. `RepeatLocusDosage` — allele tables for Phase 1 and Phase 2 (two submissions).
3. `TrgtPlvi` — PLVI per TRGT locus from every Phase 2 TRGT VCF.

Run the cells down to **Launch**, submit the four jobs, then run the rest.

## Outputs (no sample ids)

- `summaries/fig2/fig2_panel_c_rarefaction.tsv`: expected distinct alleles per locus vs haplotypes sampled, per phase (mean, IQR)
- `summaries/fig2/fig2_panel_c_examples.tsv`: alleles (dosage, haplotype count, seen in Phase 1) at the PLVI-ranked example loci
- `summaries/fig2/fig2_panel_c.summary.json`: locus set, alleles per locus, share of Phase 2 alleles absent from Phase 1
- `summaries/fig2/fig2_panel_c_trgt_concordance.summary.json`, `fig2_panel_c_dosage.phase{1,2}.summary.json`: QC

Copy them into `aou-lr-phase-2-manuscript/data/`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "aggregate_repeat_loci.py").is_file() or (_d / "terra_notebook.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
if _scripts and str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))
print("scripts:", _scripts)
for _name in ("aggregate_repeat_loci.py", "trgt_plvi.py", "compare_trgt_locus_dosage.py", "panel_c_repeat_alleles.py",
              "resolve_gl_interval_manifest.py"):
    assert (_scripts / _name).is_file(), f"{_name} not staged"

In [ ]:
import json

BUCKET = "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09"
OUT_DIR = Path("summaries/fig2").resolve()
WORK = Path("work/fig2c").resolve()
META = f"{BUCKET}/metadata/repeat_loci"
REFS = f"{BUCKET}/refs/repeat_loci"
CONFIGS = f"{BUCKET}/configs/repeat_loci"
DOCKER = "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-sv-annotation:0.1.6"

CHROMS = [f"chr{c}" for c in [*range(1, 23), "X"]]
# Phase 2: DeepVariant + GLnexus shards (GL_INTERVAL_set, one per chromosome) + integrated SV callset.
GL_NAMESPACE = "allofus-drc-wgs-LR-prodData"
GL_WORKSPACE = "AoU_DRC_LongReads_PhaseTwo_Storage"
P2_SV_BCF = "gs://prod-drc-broad/longreads/demo_group/integrated_sv_callset/v3_main.bcf"
# Phase 1: DeepVariant joint VCF + SV BCF. All four inputs are unphased.
P1_SMALL_VCF = "gs://prod-drc-broad/longreads/demo_group/phase1/Jointcall_dvcf.g.vcf.bgz.QualFT40.vcf.gz"
P1_SV_BCF = f"{BUCKET}/scratch/kvg/phase1/concat_annotated.sens_07.bcf"
SPLIT_BP = 50  # < SPLIT_BP from DeepVariant, >= SPLIT_BP from the SV callset
APPLY_FILTERS = "PASS,."

ORDER = f"{BUCKET}/metadata/fig2_discovery_order.tsv"  # Phase 2 participants (Figure 2B order)
# research_id<TAB>TRExplorer v1.0.1 TRGT VCF. tractor_mix/resources/ is gitignored, so the
# local copy exists only where it was placed by hand; otherwise read the bucket copy.
_TRGT_REL = "tractor_mix/resources/trgt_table.txt"
TRGT_TABLE = next(
    (str(p) for p in (_scripts.parent / _TRGT_REL, Path.cwd() / "aou-lr-phase-2" / _TRGT_REL, Path.home() / "aou-lr-phase-2" / _TRGT_REL) if p.is_file()),
    f"{BUCKET}/metadata/trgt_table.txt",
)
CATALOG_BED = f"{REFS}/trexplorer_v1.0.1.catalog.bed.gz"
N_CONCORDANCE = 200   # participants for the TRGT double-count check
CONCORDANCE_CHROM = "chr20"
SEED = 2026

for d in (OUT_DIR, WORK):
    d.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "GL_INTERVAL_set": f"{GL_NAMESPACE}/{GL_WORKSPACE}", "P2_SV_BCF": P2_SV_BCF,
    "P1_SMALL_VCF": P1_SMALL_VCF, "P1_SV_BCF": P1_SV_BCF, "SPLIT_BP": SPLIT_BP,
    "ORDER": ORDER, "TRGT_TABLE": TRGT_TABLE, "CATALOG_BED": CATALOG_BED, "WORK": str(WORK),
}, indent=2))

In [ ]:
import re
import shutil

assert shutil.which("bcftools") and shutil.which("gsutil"), "bcftools / gsutil not on PATH"


def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=True, text=True, **kw)


def stage(uri: str) -> Path:
    """Local path for a gs:// URI (copied once into WORK) or an existing local file."""
    if not uri.startswith("gs://"):
        return Path(uri).resolve()
    dest = WORK / Path(uri).name
    if not dest.exists():
        sh(["gsutil", "-q", "cp", uri, str(dest)])
    return dest


def gs_exists(uri: str) -> bool:
    return subprocess.run(["gsutil", "-q", "stat", uri], capture_output=True).returncode == 0


def ls_newest(pattern: str) -> dict[str, str]:
    """{basename: newest URI} for a gsutil wildcard (reruns leave several copies)."""
    res = subprocess.run(["gsutil", "ls", "-l", pattern], capture_output=True, text=True)
    best: dict[str, tuple[str, str]] = {}
    for line in res.stdout.splitlines():
        parts = line.split()
        if len(parts) == 3 and parts[2].startswith("gs://"):
            name = parts[2].rsplit("/", 1)[1]
            if name not in best or parts[1] > best[name][0]:
                best[name] = (parts[1], parts[2])
    return {k: v[1] for k, v in best.items()}


def upload_text(text: str, uri: str) -> str:
    local = WORK / uri.rsplit("/", 1)[1]
    local.write_text(text)
    sh(["gsutil", "-q", "cp", str(local), uri])
    return uri

## Catalog and participant lists

The catalog BED (chrom, start, end, TRID, motifs) comes from one TRGT VCF, which carries a
record for every TRExplorer locus. Phase 2 participants are the Figure 2B order (HPRC/HGSVC3
controls excluded). PLVI uses every participant with a TRGT VCF; the concordance subset is a random draw.

In [ ]:
import csv
import random

trgt = {}
with open(stage(TRGT_TABLE)) as fh:
    for line in fh:
        rid, uri = line.rstrip("\n").split("\t")[:2]
        trgt[rid] = uri

with open(stage(ORDER)) as fh:
    p2_ids = [r["research_id"] for r in csv.DictReader(fh, delimiter="\t")]
p2_ids = [s for s in p2_ids if not s.startswith(("HG", "NA"))]
with_trgt = sorted(s for s in p2_ids if s in trgt)
print(f"Phase 2 participants {len(p2_ids):,}; with a TRGT VCF {len(with_trgt):,}")

if not gs_exists(CATALOG_BED):
    one = stage(trgt[with_trgt[0]])
    bed = WORK / Path(CATALOG_BED).name
    sh([sys.executable, str(_scripts / "trgt_plvi.py"), "catalog", "--vcf", str(one), "--out", str(bed)])
    sh(["gsutil", "-q", "cp", str(bed), CATALOG_BED])
    one.unlink()
print("catalog:", CATALOG_BED)

rng = random.Random(SEED)
plvi_ids = with_trgt
conc_ids = sorted(rng.sample(with_trgt, min(N_CONCORDANCE, len(with_trgt))))
P2_SAMPLES = upload_text("".join(f"{s}\n" for s in p2_ids), f"{META}/phase2_participants.txt")
PLVI_LIST = upload_text("".join(f"{trgt[s]}\n" for s in plvi_ids), f"{META}/trgt_plvi_vcfs.txt")
CONC_TABLE = upload_text("".join(f"{s}\t{trgt[s]}\n" for s in conc_ids), f"{META}/trgt_concordance_subset.tsv")
print(f"PLVI VCFs {len(plvi_ids):,} ({-(-len(plvi_ids) // 50)} tasks of 50); concordance subset {len(conc_ids)}")

In [ ]:
# Phase 2 small-variant shards from GL_INTERVAL_set; SV indexes; sample-id overlap per phase.
from resolve_gl_interval_manifest import fetch_gl_interval_manifest_firecloud, normalize_gl_interval_manifest

gl = normalize_gl_interval_manifest(
    fetch_gl_interval_manifest_firecloud(GL_NAMESPACE, GL_WORKSPACE), autosomes_only=False,
    source_label=f"{GL_NAMESPACE}/{GL_WORKSPACE}",
)
gl = gl.set_index("interval_id")
missing = [c for c in CHROMS if c not in gl.index]
assert not missing, f"no GL_INTERVAL_set row for {missing}"
p2_small = {c: gl.loc[c, "VCF"] for c in CHROMS}
p2_small_idx = {c: gl.loc[c, "VCF_idx"] for c in CHROMS}
print(f"{len(p2_small)} Phase 2 GLnexus shards, e.g. {p2_small['chr20']}")


def index_of(uri: str) -> str:
    cands = (".csi", ".tbi") if uri.endswith(".bcf") else (".tbi", ".csi")
    hit = next((uri + ext for ext in cands if gs_exists(uri + ext)), None)
    assert hit, f"no index for {uri}"
    return hit


IDX = {u: index_of(u) for u in (P2_SV_BCF, P1_SV_BCF, P1_SMALL_VCF)}

# The aggregator keeps samples present in every input, so the two callsets of a phase must share ids.
# Read the VCF header with a byte range. bcftools cannot open these gs:// shards from the notebook VM,
# and the sample names are the #CHROM line, so the chromosome file itself stays in the bucket.
def vcf_ids(uri: str) -> set[str]:
    last = ""
    for n in (2_000_000, 20_000_000, 80_000_000):
        proc = subprocess.run(
            f"gsutil -q cat -r 0-{n} '{uri}' | gzip -dc 2>/dev/null || true",
            shell=True, capture_output=True,
        )
        text = proc.stdout.decode("utf-8", "replace")
        last = proc.stderr.decode("utf-8", "replace")[-500:]
        at = text.find("#CHROM\t")
        end = text.find("\n", at) if at >= 0 else -1
        if at >= 0 and end > at:
            names = text[at:end].split("\t")[9:]
            return {s for s in names if s and not s.startswith(("HG", "NA"))}
    raise SystemExit(f"no complete #CHROM line in the header of {uri}\n{last}")


for tag, small, sv in (("phase2", p2_small["chr22"], P2_SV_BCF), ("phase1", P1_SMALL_VCF, P1_SV_BCF)):
    a, b = vcf_ids(small), vcf_ids(sv)
    print(f"{tag}: small {len(a):,}  SV {len(b):,}  shared {len(a & b):,}")
    assert len(a & b) >= 0.95 * min(len(a), len(b)), f"{tag}: sample ids differ between callsets"

## Launch

Writes one input JSON per submission to `CONFIGS`. On the Terra Workflows page, open each
workflow (staged by `00_sync_repo` under `$WORKSPACE_BUCKET/wdl/repeat_loci/wdl/`), choose
"Run workflow with inputs defined by file paths", and upload the JSON.

In [ ]:
S = f"{BUCKET}/scripts"
COMMON = {"split_bp": SPLIT_BP, "apply_filters": APPLY_FILTERS, "catalog_bed": CATALOG_BED,
          "aggregate_repeat_loci_py": f"{S}/aggregate_repeat_loci.py", "docker": DOCKER}
inputs = {
    "trgt_locus_concordance": {
        **{f"TrgtLocusConcordance.{k}": v for k, v in COMMON.items()},
        "TrgtLocusConcordance.chrom": CONCORDANCE_CHROM,
        "TrgtLocusConcordance.region": f"{CONCORDANCE_CHROM}:10000000-20000000",
        "TrgtLocusConcordance.vcfs": [p2_small[CONCORDANCE_CHROM], P2_SV_BCF],
        "TrgtLocusConcordance.vcf_indexes": [p2_small_idx[CONCORDANCE_CHROM], IDX[P2_SV_BCF]],
        "TrgtLocusConcordance.vcf_roles": ["small", "sv"],
        "TrgtLocusConcordance.trgt_table": CONC_TABLE,
        "TrgtLocusConcordance.compare_trgt_locus_dosage_py": f"{S}/compare_trgt_locus_dosage.py",
    },
    "repeat_locus_dosage.phase2": {
        **{f"RepeatLocusDosage.{k}": v for k, v in COMMON.items()},
        "RepeatLocusDosage.label": "phase2",
        "RepeatLocusDosage.chroms": CHROMS,
        "RepeatLocusDosage.chrom_vcfs": [p2_small[c] for c in CHROMS],
        "RepeatLocusDosage.chrom_vcf_indexes": [p2_small_idx[c] for c in CHROMS],
        "RepeatLocusDosage.chrom_vcf_role": "small",
        "RepeatLocusDosage.genome_vcfs": [P2_SV_BCF],
        "RepeatLocusDosage.genome_vcf_indexes": [IDX[P2_SV_BCF]],
        "RepeatLocusDosage.genome_vcf_roles": ["sv"],
        "RepeatLocusDosage.samples": P2_SAMPLES,
        "RepeatLocusDosage.exclude_prefixes": "HG,NA",
    },
    "repeat_locus_dosage.phase1": {
        **{f"RepeatLocusDosage.{k}": v for k, v in COMMON.items()},
        "RepeatLocusDosage.label": "phase1",
        "RepeatLocusDosage.chroms": CHROMS,
        "RepeatLocusDosage.genome_vcfs": [P1_SMALL_VCF, P1_SV_BCF],
        "RepeatLocusDosage.genome_vcf_indexes": [IDX[P1_SMALL_VCF], IDX[P1_SV_BCF]],
        "RepeatLocusDosage.genome_vcf_roles": ["small", "sv"],
        "RepeatLocusDosage.exclude_prefixes": "HG,NA",
    },
    "trgt_plvi": {
        "TrgtPlvi.vcf_list": PLVI_LIST,
        "TrgtPlvi.catalog_bed": CATALOG_BED,
        "TrgtPlvi.shard_size": 50,
        "TrgtPlvi.min_call_rate": 0.8,
        "TrgtPlvi.label": "trgt_plvi.trexplorer_v1.0.1",
        "TrgtPlvi.trgt_plvi_py": f"{S}/trgt_plvi.py",
        "TrgtPlvi.docker": DOCKER,
    },
}
for name, body in inputs.items():
    uri = upload_text(json.dumps(body, indent=2) + "\n", f"{CONFIGS}/{name}.inputs.json")
    print(uri)

## After the runs

Finds the newest workflow outputs under `$BUCKET/submissions/`. To pin a specific run, set
the URI in `OVERRIDES`.

In [ ]:
OVERRIDES: dict[str, str] = {}


def newest_output(workflow: str, call: str, name: str) -> str:
    if name in OVERRIDES:
        return OVERRIDES[name]
    hits = ls_newest(f"{BUCKET}/submissions/*/{workflow}/*/call-{call}/**/{name}")
    assert name in hits, f"no {name} from {workflow}.{call} yet"
    return hits[name]


outputs = {
    "concordance": newest_output("TrgtLocusConcordance", "Concordance", f"{CONCORDANCE_CHROM}.trgt_concordance.summary.json"),
    "p1_alleles": newest_output("RepeatLocusDosage", "MergeChroms", "phase1.alleles.tsv.gz"),
    "p1_summary": newest_output("RepeatLocusDosage", "MergeChroms", "phase1.summary.json"),
    "p2_alleles": newest_output("RepeatLocusDosage", "MergeChroms", "phase2.alleles.tsv.gz"),
    "p2_summary": newest_output("RepeatLocusDosage", "MergeChroms", "phase2.summary.json"),
    "plvi": newest_output("TrgtPlvi", "MergePlvi", "trgt_plvi.trexplorer_v1.0.1.tsv.gz"),
}
local = {k: stage(v) for k, v in outputs.items()}
print(json.dumps(outputs, indent=2))

In [ ]:
# Step 1: double counting. Overcount = integrated sum exceeds TRGT by >= 50 bp in absolute size.
conc = json.loads(local["concordance"].read_text())
print(json.dumps(conc, indent=2))

for tag in ("p1", "p2"):
    s = json.loads(local[f"{tag}_summary"].read_text())
    c = s["counts"]
    print(tag, {k: c[k] for k in ("records_length_change", "records_assigned", "loci_polymorphic", "hap_unphased_ambiguous", "hap_possible_duplicate")},
          f"missing {s['hap_missing_frac']:.3%}  possible duplicate {s['hap_possible_duplicate_frac']:.4%}")

In [ ]:
# Steps 2 + 3: rarefaction and PLVI-ranked examples.
out = {
    "rare": OUT_DIR / "fig2_panel_c_rarefaction.tsv",
    "ex": OUT_DIR / "fig2_panel_c_examples.tsv",
    "sum": OUT_DIR / "fig2_panel_c.summary.json",
}
sh([
    sys.executable, str(_scripts / "panel_c_repeat_alleles.py"),
    "--p1-alleles", str(local["p1_alleles"]), "--p2-alleles", str(local["p2_alleles"]),
    "--plvi", str(local["plvi"]),
    "--min-call-rate", "0.9", "--n-examples", "3", "--min-alleles", "5", "--max-motif-len", "6",
    "--out-rarefaction", str(out["rare"]), "--out-examples", str(out["ex"]), "--out-summary", str(out["sum"]),
])
summary = json.loads(out["sum"].read_text())
print(json.dumps({k: summary[k] for k in ("loci", "phase1_alleles_per_locus", "phase2_alleles_per_locus",
                                          "phase2_alleles_absent_from_phase1", "examples")}, indent=2))

In [ ]:
qc = {
    "fig2_panel_c_trgt_concordance.summary.json": local["concordance"],
    "fig2_panel_c_dosage.phase1.summary.json": local["p1_summary"],
    "fig2_panel_c_dosage.phase2.summary.json": local["p2_summary"],
}
for name, src in qc.items():
    shutil.copy(src, OUT_DIR / name)
sh(["gsutil", "cp", *map(str, out.values()), *(str(OUT_DIR / n) for n in qc), f"{BUCKET}/figures/"])